# Inspect Maintain Velocity Checkpoint Structure

Load a maintain_velocity task transfer checkpoint and print out its full structure
(dictionary keys, pytree shapes, metadata, etc.).

In [1]:
%load_ext autoreload
%autoreload 2

import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import json
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import orbax.checkpoint as ocp

## Configuration

Set your checkpoint path here.

In [2]:
# UPDATE THIS to your maintain_velocity checkpoint
CHECKPOINT_PATH = "/home/mila/a/aidan.sirbu/SCAMPER/scratch/maintain_velocity_checkpoints/260209_092637_878648"
CHECKPOINT_STEP = None  # None = latest step

## 1. Discover available checkpoint steps

In [3]:
STEP_PREFIX = "TaskTransfer"

mgr_options = ocp.CheckpointManagerOptions(create=False, step_prefix=STEP_PREFIX)

with ocp.CheckpointManager(CHECKPOINT_PATH, options=mgr_options) as mgr:
    all_steps = mgr.all_steps()
    latest_step = mgr.latest_step()

print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Step prefix: {STEP_PREFIX}")
print(f"Available steps: {sorted(all_steps)}")
print(f"Latest step: {latest_step}")

step = CHECKPOINT_STEP if CHECKPOINT_STEP is not None else latest_step
print(f"\nUsing step: {step}")

Checkpoint path: /home/mila/a/aidan.sirbu/SCAMPER/scratch/maintain_velocity_checkpoints/260209_092637_878648
Step prefix: TaskTransfer
Available steps: [1, 2, 3, 4]
Latest step: 4

Using step: 4


## 2. List files on disk inside the checkpoint step directory

In [4]:
step_dir = Path(CHECKPOINT_PATH) / f"{STEP_PREFIX}_{step}"
print(f"Checkpoint step directory: {step_dir}")
print()


def print_tree(directory, prefix=""):
    """Print directory tree."""
    entries = sorted(directory.iterdir())
    for i, entry in enumerate(entries):
        is_last = i == len(entries) - 1
        connector = "\u2514\u2500\u2500 " if is_last else "\u251c\u2500\u2500 "
        if entry.is_file():
            size = entry.stat().st_size
            if size > 1024 * 1024:
                size_str = f"{size / 1024 / 1024:.1f} MB"
            elif size > 1024:
                size_str = f"{size / 1024:.1f} KB"
            else:
                size_str = f"{size} B"
            print(f"{prefix}{connector}{entry.name} ({size_str})")
        else:
            print(f"{prefix}{connector}{entry.name}/")
            child_prefix = prefix + ("    " if is_last else "\u2502   ")
            print_tree(entry, child_prefix)


if step_dir.exists():
    print_tree(step_dir)
else:
    print(f"Directory not found: {step_dir}")

Checkpoint step directory: /home/mila/a/aidan.sirbu/SCAMPER/scratch/maintain_velocity_checkpoints/260209_092637_878648/TaskTransfer_4

├── _CHECKPOINT_METADATA (577 B)
├── brax_params/
│   ├── _METADATA (6.5 KB)
│   ├── _sharding (5.1 KB)
│   ├── array_metadatas/
│   │   └── process_0 (3.1 KB)
│   ├── d/
│   │   └── 81721bf06b9443c6b1f15307d8fd6f53 (917 B)
│   ├── manifest.ocdbt (116 B)
│   └── ocdbt.process_0/
│       ├── d/
│       │   ├── 23bec074262e2074a3e56f3d51585b24 (14.8 MB)
│       │   ├── 2f05cd0a96e738425b25a488bbc53c0d (513 B)
│       │   ├── 75d60b13149555b568d9a37382f0f2c9 (12.2 MB)
│       │   ├── 91fb3164094136001d6012205a1d8cb4 (484 B)
│       │   └── a2e171fba2319077d5046d618dee4dd2 (176 B)
│       └── manifest.ocdbt (306 B)
├── config/
│   └── metadata (5.6 KB)
├── frozen_networks/
│   ├── _METADATA (16.0 KB)
│   ├── _sharding (6.7 KB)
│   ├── array_metadatas/
│   │   └── process_0 (7.7 KB)
│   ├── d/
│   │   └── d9a2fd4f53f484ba4e81e44b174b04d1 (1.9 KB)
│   ├── man

## 3. Load JSON items (config + training_metadata)

These are lightweight loads -- no array data.

In [5]:
from scamper.agent.task_transfer.maintain_velocity.checkpoint_utils import (
    load_task_transfer_metadata,
)

ckpt_info = load_task_transfer_metadata(CHECKPOINT_PATH, step=step)
config = ckpt_info["config"]
training_metadata = ckpt_info["training_metadata"]

print("=" * 60)
print("TRAINING METADATA")
print("=" * 60)
for k, v in training_metadata.items():
    print(f"  {k}: {v}")

TRAINING METADATA
  completed_timesteps: 61603840
  eval_step: 4
  wandb_run_id: New3_S0.3_prior_decoder_260209_092637_878648
  run_id: 260209_092637_878648
  mode: prior_decoder


In [6]:
print("=" * 60)
print("CONFIG (top-level keys)")
print("=" * 60)
for k, v in config.items():
    if isinstance(v, dict):
        print(f"  {k}: <dict with {len(v)} keys: {list(v.keys())}>")
    else:
        print(f"  {k}: {v}")

CONFIG (top-level keys)
  mode: prior_decoder
  checkpoint: <dict with 2 keys: ['path', 'step']>
  decoder_only: <dict with 1 keys: ['_placeholder']>
  prior_decoder: <dict with 2 keys: ['deterministic_prior', 'noise_logvar']>
  scratch: <dict with 1 keys: ['_placeholder']>
  network: <dict with 2 keys: ['policy_layers', 'value_layers']>
  train: <dict with 19 keys: ['num_timesteps', 'episode_length', 'num_envs', 'batch_size', 'num_minibatches', 'num_updates_per_batch', 'learning_rate', 'entropy_cost', 'discounting', 'clipping_epsilon', 'gae_lambda', 'max_grad_norm', 'vf_loss_coefficient', 'reward_scaling', 'normalize_observations', 'unroll_length', 'action_repeat', 'seed', 'eval_every']>
  env: <dict with 5 keys: ['xml_variant', 'mujoco_impl', 'init_z', 'target_speed', 'reward_weights']>
  checkpointing: <dict with 3 keys: ['enabled', 'model_path', 'max_to_keep']>
  logging: <dict with 8 keys: ['project', 'group_name', 'exp_name', 'render_camera', 'render_fps', 'render_height', 'rende

In [7]:
print("=" * 60)
print("CONFIG - full JSON")
print("=" * 60)
print(json.dumps(config, indent=2, default=str))

CONFIG - full JSON
{
  "mode": "prior_decoder",
  "checkpoint": {
    "path": "/home/mila/a/aidan.sirbu/SCAMPER/scratch/prior_checkpoints/golden_rodent_prior_jax",
    "step": 23
  },
  "decoder_only": {
    "_placeholder": true
  },
  "prior_decoder": {
    "deterministic_prior": true,
    "noise_logvar": -2.0
  },
  "scratch": {
    "_placeholder": true
  },
  "network": {
    "policy_layers": [
      2048,
      1024,
      512
    ],
    "value_layers": [
      2048,
      1024,
      1024,
      512
    ]
  },
  "train": {
    "num_timesteps": 2000000000,
    "episode_length": 2000,
    "num_envs": 1024,
    "batch_size": 1024,
    "num_minibatches": 16,
    "num_updates_per_batch": 4,
    "learning_rate": 0.0001,
    "entropy_cost": 0.0,
    "discounting": 0.99,
    "clipping_epsilon": 0.3,
    "gae_lambda": 0.95,
    "max_grad_norm": 50.0,
    "vf_loss_coefficient": 0.5,
    "reward_scaling": 1.0,
    "normalize_observations": true,
    "unroll_length": 20,
    "action_repeat": 

## 4. Load frozen_networks and inspect structure

The `frozen_networks` item contains the encoder, prior, decoder params and the
imitation normalizer from the source mlp_prior checkpoint. We load these with
`StandardRestore` using an abstract pytree built from the config.

In [8]:
from scamper.agent.task_transfer.maintain_velocity.checkpoint_utils import (
    load_prior_checkpoint,
)

# Build abstract frozen_networks from the source prior checkpoint
prior_checkpoint_path = config["checkpoint"]["path"]
prior_checkpoint_step = config["checkpoint"]["step"]

print(f"Source prior checkpoint: {prior_checkpoint_path}")
print(f"Source prior step: {prior_checkpoint_step}")

(
    encoder_params,
    prior_params,
    decoder_params,
    normalizer_params,
    source_ckpt_cfg,
) = load_prior_checkpoint(prior_checkpoint_path, prior_checkpoint_step)

# Build the same frozen_networks dict that was saved
frozen_networks = {
    "encoder_params": encoder_params,
    "prior_params": prior_params,
    "decoder_params": decoder_params,
    "imitation_normalizer": normalizer_params,
}

# Now load the actual frozen_networks from the task transfer checkpoint
with ocp.CheckpointManager(CHECKPOINT_PATH, options=mgr_options) as mgr:
    frozen_data = mgr.restore(
        step,
        args=ocp.args.Composite(
            frozen_networks=ocp.args.StandardRestore(frozen_networks),
        ),
    )

frozen = frozen_data["frozen_networks"]
print(f"\nLoaded frozen_networks from task transfer checkpoint")

Source prior checkpoint: /home/mila/a/aidan.sirbu/SCAMPER/scratch/prior_checkpoints/golden_rodent_prior_jax
Source prior step: 23

Loaded frozen_networks from task transfer checkpoint


In [9]:
def print_pytree(tree, prefix="", name="root"):
    """Recursively print a JAX pytree showing keys, types, and array shapes."""
    if isinstance(tree, dict):
        print(f"{prefix}{name}: <dict with {len(tree)} keys>")
        items = list(tree.items())
        for i, (k, v) in enumerate(items):
            is_last = i == len(items) - 1
            connector = "\u2514\u2500 " if is_last else "\u251c\u2500 "
            child_prefix = prefix + ("   " if is_last else "\u2502  ")
            print_pytree(v, prefix=prefix + connector, name=str(k))
    elif isinstance(tree, (list, tuple)):
        type_name = type(tree).__name__
        print(f"{prefix}{name}: <{type_name} with {len(tree)} elements>")
        for i, v in enumerate(tree):
            is_last = i == len(tree) - 1
            connector = "\u2514\u2500 " if is_last else "\u251c\u2500 "
            print_pytree(v, prefix=prefix + connector, name=f"[{i}]")
    elif hasattr(tree, "__dict__") and not isinstance(tree, (jnp.ndarray, np.ndarray)):
        # Named tuple or dataclass-like
        type_name = type(tree).__name__
        fields = {k: v for k, v in tree.__dict__.items() if not k.startswith("_")}
        if not fields and hasattr(tree, "_fields"):
            fields = {f: getattr(tree, f) for f in tree._fields}
        if not fields and hasattr(tree, "_asdict"):
            fields = tree._asdict()
        print(f"{prefix}{name}: <{type_name} with {len(fields)} fields>")
        items = list(fields.items())
        for i, (k, v) in enumerate(items):
            is_last = i == len(items) - 1
            connector = "\u2514\u2500 " if is_last else "\u251c\u2500 "
            print_pytree(v, prefix=prefix + connector, name=str(k))
    elif isinstance(tree, (jnp.ndarray, np.ndarray)):
        print(
            f"{prefix}{name}: {type(tree).__name__} shape={tree.shape} dtype={tree.dtype}"
        )
    else:
        print(f"{prefix}{name}: {type(tree).__name__} = {tree}")

In [10]:
print("=" * 60)
print("FROZEN NETWORKS STRUCTURE")
print("=" * 60)
print_pytree(frozen, name="frozen_networks")

FROZEN NETWORKS STRUCTURE
frozen_networks: <dict with 4 keys>
├─ decoder_params: <dict with 7 keys>
├─ ├─ LayerNorm_0: <dict with 2 keys>
├─ ├─ ├─ bias: ArrayImpl shape=(1024,) dtype=float32
├─ ├─ └─ scale: ArrayImpl shape=(1024,) dtype=float32
├─ ├─ LayerNorm_1: <dict with 2 keys>
├─ ├─ ├─ bias: ArrayImpl shape=(1024,) dtype=float32
├─ ├─ └─ scale: ArrayImpl shape=(1024,) dtype=float32
├─ ├─ LayerNorm_2: <dict with 2 keys>
├─ ├─ ├─ bias: ArrayImpl shape=(512,) dtype=float32
├─ ├─ └─ scale: ArrayImpl shape=(512,) dtype=float32
├─ ├─ hidden_0: <dict with 2 keys>
├─ ├─ ├─ bias: ArrayImpl shape=(1024,) dtype=float32
├─ ├─ └─ kernel: ArrayImpl shape=(293, 1024) dtype=float32
├─ ├─ hidden_1: <dict with 2 keys>
├─ ├─ ├─ bias: ArrayImpl shape=(1024,) dtype=float32
├─ ├─ └─ kernel: ArrayImpl shape=(1024, 1024) dtype=float32
├─ ├─ hidden_2: <dict with 2 keys>
├─ ├─ ├─ bias: ArrayImpl shape=(512,) dtype=float32
├─ ├─ └─ kernel: ArrayImpl shape=(1024, 512) dtype=float32
├─ └─ hidden_3: <dict with

## 5. Load brax_params (PPO training state) and inspect structure

The `brax_params` item is a tuple of `(normalizer_params, policy_params, value_params)`.
We need to build an abstract pytree with the correct shape to restore it.

In [11]:
import functools

from brax.training import distribution
from brax.training.acme import running_statistics, specs
from brax.training.agents.ppo import networks as ppo_networks

from scamper.agent.task_transfer.maintain_velocity.checkpoint_utils import (
    load_task_transfer_brax_params,
    make_decoder_inference_fn,
    make_prior_inference_fn,
)
from scamper.agent.task_transfer.maintain_velocity.wrappers import (
    DecoderHighLevelWrapper,
    FlatObsWrapper,
    PriorDecoderHighLevelWrapper,
)
from scamper.agent.task_transfer.maintain_velocity.scratch_networks import (
    make_scratch_ppo_networks,
)
from vnl_playground.tasks.rodent import consts as rodent_consts
from vnl_playground.tasks.rodent import maintain_velocity

mode = config["mode"]
ckpt_cfg = config["checkpoint_config"]

latent_size = ckpt_cfg["network_config"]["intention_size"]
proprio_size = ckpt_cfg["network_config"]["obs_sizes"]["proprioception"]
action_size = ckpt_cfg["network_config"]["action_size"]

print(f"Mode: {mode}")
print(f"Latent size: {latent_size}")
print(f"Proprio size: {proprio_size}")
print(f"Action size: {action_size}")

Mode: prior_decoder
Latent size: 16
Proprio size: 277
Action size: 38


In [12]:
# Create a minimal environment to get observation size
env_cfg = maintain_velocity.default_config()

xml_variant = config["env"].get("xml_variant", "full-collisions")
if xml_variant == "limited-collisions":
    env_cfg.walker_xml_path = rodent_consts.RODENT_XML_PATH
elif xml_variant == "full-collisions":
    env_cfg.walker_xml_path = rodent_consts.RODENT_FULL_COLLISION_XML
elif xml_variant == "no-tail-collisions":
    env_cfg.walker_xml_path = rodent_consts.RODENT_NO_TAIL_COLLISION_XML

env_cfg.mujoco_impl = config["env"].get("mujoco_impl", "jax")
env_cfg.ctrl_dt = ckpt_cfg["env_config"]["ctrl_dt"]
env_cfg.init_z = config["env"]["init_z"]
env_cfg.target_speed = config["env"]["target_speed"]

# Create inference functions for wrappers
decoder_fn = make_decoder_inference_fn(decoder_params, normalizer_params, ckpt_cfg)
base_env = maintain_velocity.MaintainVelocity(config=env_cfg)

if mode == "decoder_only":
    env = DecoderHighLevelWrapper(base_env, decoder_fn, latent_size, proprio_size)
elif mode == "scratch":
    env = FlatObsWrapper(base_env)
elif mode == "prior_decoder":
    prior_fn = make_prior_inference_fn(prior_params, normalizer_params, ckpt_cfg)
    env = PriorDecoderHighLevelWrapper(
        base_env,
        prior_fn,
        decoder_fn,
        latent_size,
        proprio_size,
        deterministic_prior=config["prior_decoder"]["deterministic_prior"],
        noise_logvar=config["prior_decoder"]["noise_logvar"],
    )

# Get obs size from a reset
jit_reset = jax.jit(env.reset)
start_state = jit_reset(jax.random.PRNGKey(0))
obs_size = start_state.obs.shape[-1]
policy_action_size = env.action_size

print(f"Observation size: {obs_size}")
print(f"Policy action size: {policy_action_size}")

Warp 1.11.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Quadro RTX 8000" (44 GiB, sm_75, mempool enabled)
   Kernel cache:
     /home/mila/a/aidan.sirbu/.cache/warp/1.11.0
Warp DeprecationWarning: The symbol `warp.types.warp_type_to_np_dtype` will soon be removed from the public API. It can still be accessed from `warp._src.types.warp_type_to_np_dtype` but might be changed or removed without notice.
Module mujoco.mjx.third_party.mujoco_warp._src.smooth e142b0f load on device 'cuda:0' took 26.14 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver de6dc98 load on device 'cuda:0' took 9.60 ms  (cached)
Module _nxn_broadphase__locals__kernel_8c837afb 8c837af load on device 'cuda:0' took 6.52 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_3a95495c 42367f3 load on device 'cuda:0' took 11.28 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 51cd7b3 load on devi

E0212 10:04:27.048852  997722 slow_operation_alarm.cc:73] Constant folding an instruction is taking > 1s:

  %reduce_sum.91 = s32[] reduce(%constant.82, %constant.181), dimensions={0}, to_apply=%region_0.4, metadata={op_name="jit(reset)/reduce_sum" stack_frame_id=47}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
E0212 10:04:41.659609  996647 slow_operation_alarm.cc:140] The operation took 15.610867488s
Constant folding an instruction is taking > 1s:

  %reduce_sum.91 = s32[] reduce(%constant.82, %constant.181), dimensions={0}, to_apply=%region_0.4, metadata={op_name="jit(reset)/reduce_sum" stack_frame_id=47}

This isn't necessarily a bug

Observation size: 331
Policy action size: 16


In [13]:
# Build abstract brax_params
normalize = running_statistics.normalize

if mode == "scratch":
    task_obs_size = obs_size - proprio_size
    decoder_hidden_layer_sizes = tuple(
        ckpt_cfg["network_config"]["decoder_layer_sizes"]
    )

    dummy_net = make_scratch_ppo_networks(
        task_obs_size=task_obs_size,
        proprio_size=proprio_size,
        action_size=action_size,
        latent_size=latent_size,
        policy_hidden_layer_sizes=tuple(config["network"]["policy_layers"]),
        decoder_hidden_layer_sizes=decoder_hidden_layer_sizes,
        value_hidden_layer_sizes=tuple(config["network"]["value_layers"]),
    )
    key_p, key_v = jax.random.split(jax.random.PRNGKey(0))
    dummy_normalizer = running_statistics.init_state(
        specs.Array(start_state.obs.shape[-1:], jnp.dtype("float32"))
    )
    dummy_policy = dummy_net.policy_network.init(key_p)
    dummy_value = dummy_net.value_network.init(key_v)
else:
    dummy_net = ppo_networks.make_ppo_networks(
        obs_size,
        policy_action_size,
        preprocess_observations_fn=normalize,
        policy_hidden_layer_sizes=tuple(config["network"]["policy_layers"]),
        value_hidden_layer_sizes=tuple(config["network"]["value_layers"]),
    )
    key_p, key_v = jax.random.split(jax.random.PRNGKey(0))
    dummy_normalizer = running_statistics.init_state(
        specs.Array(start_state.obs.shape[-1:], jnp.dtype("float32"))
    )
    dummy_policy = dummy_net.policy_network.init(key_p)
    dummy_value = dummy_net.value_network.init(key_v)

abstract_brax_params = (dummy_normalizer, dummy_policy, dummy_value)

# Load the actual brax_params
brax_params = load_task_transfer_brax_params(
    CHECKPOINT_PATH, abstract_brax_params, step=step
)

print(f"Loaded brax_params (tuple of {len(brax_params)} elements)")

Loaded brax_params (tuple of 3 elements)


In [14]:
print("=" * 60)
print("BRAX PARAMS STRUCTURE")
print("=" * 60)

normalizer_params_loaded, policy_params, value_params = brax_params

print("\n--- [0] normalizer_params ---")
print_pytree(normalizer_params_loaded, name="normalizer_params")

print("\n--- [1] policy_params ---")
print_pytree(policy_params, name="policy_params")

print("\n--- [2] value_params ---")
print_pytree(value_params, name="value_params")

BRAX PARAMS STRUCTURE

--- [0] normalizer_params ---
normalizer_params: <RunningStatisticsState with 6 fields>
├─ mean: ArrayImpl shape=(331,) dtype=float32
├─ std: ArrayImpl shape=(331,) dtype=float32
├─ count: <UInt64 with 2 fields>
├─ ├─ hi: ArrayImpl shape=() dtype=uint32
├─ └─ lo: ArrayImpl shape=() dtype=uint32
├─ summed_variance: ArrayImpl shape=(331,) dtype=float32
├─ std_eps: float = 0.0
└─ mode: <NormalizationMode with 0 fields>

--- [1] policy_params ---
policy_params: <dict with 1 keys>
└─ params: <dict with 4 keys>
└─ ├─ hidden_0: <dict with 2 keys>
└─ ├─ ├─ bias: ArrayImpl shape=(2048,) dtype=float32
└─ ├─ └─ kernel: ArrayImpl shape=(331, 2048) dtype=float32
└─ ├─ hidden_1: <dict with 2 keys>
└─ ├─ ├─ bias: ArrayImpl shape=(1024,) dtype=float32
└─ ├─ └─ kernel: ArrayImpl shape=(2048, 1024) dtype=float32
└─ ├─ hidden_2: <dict with 2 keys>
└─ ├─ ├─ bias: ArrayImpl shape=(512,) dtype=float32
└─ ├─ └─ kernel: ArrayImpl shape=(1024, 512) dtype=float32
└─ └─ hidden_3: <dict wit

## 6. Full checkpoint summary

Summary of all four items saved in the checkpoint.

In [ ]:
def count_params(tree):
    """Count total number of parameters in a pytree."""
    leaves = jax.tree_util.tree_leaves(tree)
    return sum(x.size for x in leaves if hasattr(x, "size"))


print("=" * 60)
print(f"CHECKPOINT SUMMARY (step {step})")
print("=" * 60)

print(f"\nCheckpoint path: {CHECKPOINT_PATH}")
print(f"Mode: {mode}")
print(f"Completed timesteps: {training_metadata.get('completed_timesteps', 'N/A')}")
print(f"Eval step: {training_metadata.get('eval_step', 'N/A')}")
print(f"Run ID: {training_metadata.get('run_id', 'N/A')}")
print(f"W&B run ID: {training_metadata.get('wandb_run_id', 'N/A')}")

print(f"\n--- Top-level checkpoint items ---")
print(
    f"  1. config          (JSON)  - Training config with {len(config)} top-level keys"
)
print(f"  2. training_metadata (JSON) - {len(training_metadata)} fields")
print(f"  3. frozen_networks (arrays) - {len(frozen)} keys: {list(frozen.keys())}")
print(f"  4. brax_params     (arrays) - tuple of 3: (normalizer, policy, value)")

print(f"\n--- Parameter counts ---")
print(f"  Encoder:    {count_params(frozen['encoder_params']):>12,}")
print(f"  Prior:      {count_params(frozen['prior_params']):>12,}")
print(f"  Decoder:    {count_params(frozen['decoder_params']):>12,}")
print(f"  Policy:     {count_params(policy_params):>12,}")
print(f"  Value:      {count_params(value_params):>12,}")
print(f"  " + "-" * 26)
total = (
    count_params(frozen["encoder_params"])
    + count_params(frozen["prior_params"])
    + count_params(frozen["decoder_params"])
    + count_params(policy_params)
    + count_params(value_params)
)
print(f"  Total:      {total:>12,}")